In [ ]:
import json
import time
import random
from faker import Faker
import logging
from datetime import datetime, timedelta
import numpy as np

fake = Faker(['en_US', 'es_ES', 'fr_FR', 'de_DE', 'ja_JP'])

# Industry-specific datasets
RETAIL_POSTS = {
    'positive': [
        "Just received my order and the quality is outstanding! Fast shipping too 📦✨",
        "Customer service helped me find the perfect size. Amazing experience! 🛍️",
        "This brand never disappoints. Worth every penny! Highly recommend 💯",
        "The new collection is gorgeous! Already planning my next purchase 😍"
    ],
    'negative': [
        "Ordered 2 weeks ago and still nothing. Customer service ignores my emails 😡",
        "Product looks nothing like the website photos. Very misleading 😤",
        "Expensive for such poor quality. Returning everything 📉",
        "Sizing is completely wrong. Waste of money 👎"
    ],
    'mixed': [
        "Love the design but quality could be better for this price point 🤔",
        "Great customer service but shipping took forever. Mixed feelings 📦⏰",
        "Product is okay, not great but not terrible either. Average experience 🤷"
    ]
}

FINANCE_POSTS = {
    'positive': [
        "This investment app made portfolio management so much easier! Great UX 📈",
        "Finally found a bank with reasonable fees. Switching was worth it 💰",
        "Their financial advisor gave excellent advice. Portfolio is growing! 📊"
    ],
    'negative': [
        "Hidden fees everywhere! This bank is a scam. Moving my money elsewhere 😠",
        "App crashes during important trades. Lost money due to technical issues 📱💸",
        "Customer service has no clue about their own products. Frustrated! 🤦"
    ]
}

HEALTHCARE_POSTS = {
    'positive': [
        "Dr. Smith explained everything clearly and made me feel comfortable 👩‍⚕️",
        "This health app tracks my symptoms perfectly. Very helpful for managing my condition 📱💊",
        "Hospital staff was incredibly caring during my stay. Thank you! 🏥❤️"
    ],
    'negative': [
        "Waited 3 hours for a 10-minute appointment. This healthcare system is broken 🕐😤",
        "Insurance denied my claim again. Fighting for basic coverage is exhausting 📄💸",
        "Side effects from this medication are worse than the original problem 💊😷"
    ]
}

# Emotion categories with intensity
EMOTIONS = {
    'joy': {'words': ['happy', 'excited', 'thrilled', 'delighted', 'ecstatic'], 'intensity': (70, 95)},
    'anger': {'words': ['furious', 'annoyed', 'frustrated', 'outraged', 'livid'], 'intensity': (60, 90)},
    'fear': {'words': ['worried', 'anxious', 'scared', 'terrified', 'nervous'], 'intensity': (50, 85)},
    'sadness': {'words': ['disappointed', 'upset', 'depressed', 'heartbroken', 'devastated'], 'intensity': (40, 80)},
    'surprise': {'words': ['shocked', 'amazed', 'stunned', 'astonished', 'bewildered'], 'intensity': (30, 70)},
    'disgust': {'words': ['disgusted', 'revolted', 'repulsed', 'sickened', 'appalled'], 'intensity': (50, 85)}
}

class AdvancedDataGenerator:
    def __init__(self):
        self.user_profiles = self.generate_user_profiles(1000)
        self.trends = self.generate_trends()
        
    def generate_user_profiles(self, count):
        profiles = {}
        for _ in range(count):
            user_id = fake.user_name()
            profiles[user_id] = {
                'age_group': random.choice(['18-25', '26-35', '36-45', '46-55', '55+']),
                'interests': random.sample(['tech', 'fashion', 'food', 'travel', 'sports', 'finance', 'health'], 3),
                'sentiment_bias': random.uniform(-0.3, 0.3),  # Some users are more positive/negative
                'language': random.choice(['en', 'es', 'fr', 'de', 'ja']),
                'location': fake.city(),
                'follower_count': random.randint(50, 50000),
                'verified': random.random() < 0.1,
                'account_created': fake.date_between(start_date='-5y', end_date='today')
            }
        return profiles
    
    def generate_trends(self):
        """Simulate trending topics that affect sentiment"""
        return {
            'current_events': [
                {'topic': 'economic_downturn', 'sentiment_impact': -0.4, 'probability': 0.1},
                {'topic': 'new_product_launch', 'sentiment_impact': 0.3, 'probability': 0.15},
                {'topic': 'celebrity_scandal', 'sentiment_impact': -0.2, 'probability': 0.05},
                {'topic': 'holiday_season', 'sentiment_impact': 0.2, 'probability': 0.2}
            ],
            'seasonal_patterns': {
                'monday': -0.1,  # Monday blues
                'friday': 0.2,   # TGIF
                'weekend': 0.15,
                'holiday': 0.3
            }
        }
    
    def detect_sarcasm_indicators(self, text):
        """Simple sarcasm detection based on patterns"""
        sarcasm_indicators = [
            'oh great', 'just perfect', 'wonderful', 'fantastic',
            'exactly what I needed', 'thanks a lot', 'brilliant'
        ]
        
        # Check for positive words with negative context
        positive_with_negative = any(indicator in text.lower() for indicator in sarcasm_indicators)
        excessive_punctuation = '!!!' in text or '...' in text
        
        return positive_with_negative or excessive_punctuation
    
    def extract_aspects(self, text, domain='retail'):
        """Extract aspects being discussed (product, service, price, etc.)"""
        aspect_keywords = {
            'retail': {
                'product': ['quality', 'design', 'material', 'color', 'size', 'fit'],
                'service': ['customer service', 'support', 'help', 'staff'],
                'shipping': ['delivery', 'shipping', 'package', 'arrived'],
                'price': ['price', 'cost', 'expensive', 'cheap', 'value', 'money']
            },
            'finance': {
                'fees': ['fee', 'charge', 'cost', 'expensive'],
                'service': ['customer service', 'advisor', 'support'],
                'app': ['app', 'website', 'platform', 'interface'],
                'rates': ['rate', 'interest', 'return', 'profit']
            }
        }
        
        detected_aspects = {}
        if domain in aspect_keywords:
            for aspect, keywords in aspect_keywords[domain].items():
                if any(keyword in text.lower() for keyword in keywords):
                    detected_aspects[aspect] = random.uniform(0.1, 0.9)  # Aspect relevance score
        
        return detected_aspects
    
    def generate_realistic_post(self, user_id=None, domain='retail'):
        """Generate a realistic social media post with rich metadata"""
        if user_id is None:
            user_id = random.choice(list(self.user_profiles.keys()))
        
        user = self.user_profiles[user_id]
        
        # Choose post type based on domain
        domain_posts = {
            'retail': RETAIL_POSTS,
            'finance': FINANCE_POSTS,
            'healthcare': HEALTHCARE_POSTS
        }.get(domain, RETAIL_POSTS)
        
        # Apply user sentiment bias and current trends
        sentiment_modifier = user['sentiment_bias']
        for trend in self.trends['current_events']:
            if random.random() < trend['probability']:
                sentiment_modifier += trend['sentiment_impact']
        
        # Choose sentiment category based on bias
        if sentiment_modifier > 0.2:
            sentiment_type = 'positive'
        elif sentiment_modifier < -0.2:
            sentiment_type = 'negative'
        else:
            sentiment_type = random.choice(['positive', 'negative', 'mixed'])
        
        base_text = random.choice(domain_posts[sentiment_type])
        
        # Add realistic variations
        if random.random() < 0.3:  # 30% chance of adding hashtags
            hashtags = ['#' + interest for interest in random.sample(user['interests'], 2)]
            base_text += ' ' + ' '.join(hashtags)
        
        if random.random() < 0.2:  # 20% chance of mentioning other users
            base_text += f' @{fake.user_name()}'
        
        # Detect emotions and their intensities
        primary_emotion = self.detect_primary_emotion(base_text)
        emotion_scores = self.calculate_emotion_scores(base_text, primary_emotion)
        
        # Detect sarcasm
        is_sarcastic = self.detect_sarcasm_indicators(base_text)
        
        # Extract aspects being discussed
        aspects = self.extract_aspects(base_text, domain)
        
        return {
            'id': fake.uuid4(),
            'user_id': user_id,
            'text': base_text,
            'timestamp': time.time(),
            'domain': domain,
            
            # User context
            'user_profile': {
                'age_group': user['age_group'],
                'follower_count': user['follower_count'],
                'verified': user['verified'],
                'location': user['location'],
                'language': user['language']
            },
            
            # Enhanced sentiment analysis
            'true_sentiment': {
                'polarity': sentiment_type,
                'confidence': random.uniform(0.6, 0.95),
                'emotions': emotion_scores,
                'primary_emotion': primary_emotion,
                'is_sarcastic': is_sarcastic,
                'aspects': aspects
            },
            
            # Engagement metrics (for weighting)
            'engagement': {
                'likes': random.randint(0, 1000),
                'shares': random.randint(0, 100),
                'comments': random.randint(0, 50),
                'reach': random.randint(user['follower_count'], user['follower_count'] * 3)
            },
            
            # Content features
            'features': {
                'text_length': len(base_text),
                'has_emojis': bool(any(char in base_text for char in '😀😃😄😁😊😍😭😡🤔👍👎💯')),
                'has_hashtags': '#' in base_text,
                'has_mentions': '@' in base_text,
                'has_urls': 'http' in base_text,
                'sentiment_words_count': self.count_sentiment_words(base_text)
            }
        }
    
    def detect_primary_emotion(self, text):
        """Detect the primary emotion in the text"""
        text_lower = text.lower()
        emotion_scores = {}
        
        for emotion, data in EMOTIONS.items():
            score = sum(1 for word in data['words'] if word in text_lower)
            emotion_scores[emotion] = score
        
        return max(emotion_scores, key=emotion_scores.get) if max(emotion_scores.values()) > 0 else 'neutral'
    
    def calculate_emotion_scores(self, text, primary_emotion):
        """Calculate intensity scores for all emotions"""
        scores = {}
        text_lower = text.lower()
        
        for emotion, data in EMOTIONS.items():
            base_score = sum(10 for word in data['words'] if word in text_lower)
            
            if emotion == primary_emotion:
                base_score *= 2  # Boost primary emotion
            
            # Add some randomness and normalize
            intensity = min(100, base_score + random.randint(-10, 10))
            scores[emotion] = max(0, intensity)
        
        return scores
    
    def count_sentiment_words(self, text):
        """Count positive and negative sentiment words"""
        positive_words = ['good', 'great', 'excellent', 'amazing', 'love', 'perfect', 'best']
        negative_words = ['bad', 'terrible', 'awful', 'hate', 'worst', 'horrible', 'disappointing']
        
        text_lower = text.lower()
        positive_count = sum(1 for word in positive_words if word in text_lower)
        negative_count = sum(1 for word in negative_words if word in text_lower)
        
        return {'positive': positive_count, 'negative': negative_count}



{'id': '0cefd079-44a9-4d48-82af-49aaa75e0725', 'user_id': 'fioljuanita', 'text': 'This brand never disappoints. Worth every penny! Highly recommend 💯', 'timestamp': 1757928353.1624649, 'domain': 'retail', 'user_profile': {'age_group': '36-45', 'follower_count': 23048, 'verified': False, 'location': 'Neunburg vorm Wald', 'language': 'es'}, 'true_sentiment': {'polarity': 'positive', 'confidence': 0.7194403475127943, 'emotions': {'joy': 0, 'anger': 0, 'fear': 10, 'sadness': 4, 'surprise': 0, 'disgust': 4}, 'primary_emotion': 'neutral', 'is_sarcastic': False, 'aspects': {}}, 'engagement': {'likes': 866, 'shares': 82, 'comments': 32, 'reach': 42625}, 'features': {'text_length': 67, 'has_emojis': True, 'has_hashtags': False, 'has_mentions': False, 'has_urls': False, 'sentiment_words_count': {'positive': 0, 'negative': 0}}}


ERROR:root:Error: 'mixed'


{'id': '3b064f3f-5d93-4b3c-b456-c38fb5d9b554', 'user_id': 'walterhecker', 'text': 'This brand never disappoints. Worth every penny! Highly recommend 💯', 'timestamp': 1757928359.1694045, 'domain': 'retail', 'user_profile': {'age_group': '55+', 'follower_count': 43917, 'verified': False, 'location': 'Klein', 'language': 'ja'}, 'true_sentiment': {'polarity': 'positive', 'confidence': 0.6239642222830496, 'emotions': {'joy': 1, 'anger': 7, 'fear': 0, 'sadness': 0, 'surprise': 1, 'disgust': 0}, 'primary_emotion': 'neutral', 'is_sarcastic': False, 'aspects': {}}, 'engagement': {'likes': 568, 'shares': 51, 'comments': 38, 'reach': 119783}, 'features': {'text_length': 67, 'has_emojis': True, 'has_hashtags': False, 'has_mentions': False, 'has_urls': False, 'sentiment_words_count': {'positive': 0, 'negative': 0}}}
{'id': '64ca7df8-f95a-420a-a552-c96f5bad5ef9', 'user_id': 'ophillips', 'text': 'Finally found a bank with reasonable fees. Switching was worth it 💰 #health #sports', 'timestamp': 175792

ERROR:root:Error: 'mixed'


{'id': 'c2d32877-b444-4244-b587-c1f7079a7e6e', 'user_id': 'nicolasfrederique', 'text': 'Ordered 2 weeks ago and still nothing. Customer service ignores my emails 😡', 'timestamp': 1757928369.1781995, 'domain': 'retail', 'user_profile': {'age_group': '46-55', 'follower_count': 8226, 'verified': False, 'location': 'Sainte Colette', 'language': 'es'}, 'true_sentiment': {'polarity': 'negative', 'confidence': 0.6022915764573316, 'emotions': {'joy': 1, 'anger': 0, 'fear': 0, 'sadness': 4, 'surprise': 0, 'disgust': 0}, 'primary_emotion': 'neutral', 'is_sarcastic': False, 'aspects': {'service': 0.3469300050328613}}, 'engagement': {'likes': 874, 'shares': 46, 'comments': 20, 'reach': 24586}, 'features': {'text_length': 75, 'has_emojis': True, 'has_hashtags': False, 'has_mentions': False, 'has_urls': False, 'sentiment_words_count': {'positive': 0, 'negative': 0}}}
{'id': '6ff49770-3d21-49ec-acc2-aa0a69e1efba', 'user_id': 'suzukimaaya', 'text': 'Finally found a bank with reasonable fees. Switching

...

ERROR:root:Error: 'mixed'


{'id': '74c0b0a0-f5fc-4860-9c7e-8800ae4fb5c5', 'user_id': 'heleneaustermuehle', 'text': 'Customer service helped me find the perfect size. Amazing experience! 🛍️', 'timestamp': 1757928430.2455933, 'domain': 'retail', 'user_profile': {'age_group': '18-25', 'follower_count': 24558, 'verified': False, 'location': '長生郡白子町', 'language': 'en'}, 'true_sentiment': {'polarity': 'positive', 'confidence': 0.9426388692953804, 'emotions': {'joy': 3, 'anger': 0, 'fear': 0, 'sadness': 0, 'surprise': 4, 'disgust': 0}, 'primary_emotion': 'neutral', 'is_sarcastic': False, 'aspects': {'product': 0.15597869800529454, 'service': 0.28905046679674806}}, 'engagement': {'likes': 569, 'shares': 37, 'comments': 35, 'reach': 33462}, 'features': {'text_length': 72, 'has_emojis': False, 'has_hashtags': False, 'has_mentions': False, 'has_urls': False, 'sentiment_words_count': {'positive': 2, 'negative': 0}}}
{'id': '40615b31-dd1f-44b4-806f-5ee10c6fcc51', 'user_id': 'bonniewilliams', 'text': 'Finally found a bank wit

ERROR:root:Error: 'mixed'


{'id': 'd0079804-042e-4d6b-815a-d900285b46ed', 'user_id': 'dyoshida', 'text': 'Expensive for such poor quality. Returning everything 📉 #health #tech', 'timestamp': 1757928437.2496626, 'domain': 'retail', 'user_profile': {'age_group': '36-45', 'follower_count': 31176, 'verified': False, 'location': 'Mathieu-sur-Payet', 'language': 'ja'}, 'true_sentiment': {'polarity': 'negative', 'confidence': 0.7233864726797821, 'emotions': {'joy': 4, 'anger': 0, 'fear': 0, 'sadness': 10, 'surprise': 0, 'disgust': 5}, 'primary_emotion': 'neutral', 'is_sarcastic': False, 'aspects': {'product': 0.6423355214764452, 'price': 0.7767903357261818}}, 'engagement': {'likes': 42, 'shares': 0, 'comments': 27, 'reach': 83863}, 'features': {'text_length': 69, 'has_emojis': False, 'has_hashtags': True, 'has_mentions': False, 'has_urls': False, 'sentiment_words_count': {'positive': 0, 'negative': 0}}}
{'id': '2617964c-9eac-44de-9cfa-ae55613233af', 'user_id': 'ekkehardhornich', 'text': 'Customer service has no clue ab

KeyboardInterrupt: 

In [6]:
generator = AdvancedDataGenerator()
user_posts = []



domains = ['retail', 'finance', 'healthcare']

logging.info("Starting advanced data generator...")

while True:
    try:
        # Generate posts for different domains
        for domain in domains:
            post = generator.generate_realistic_post(domain=domain)
            user_posts.append(post)
            print(post)
            logging.info(f"Sent {domain} post: {post['text'][:50]}...")
            time.sleep(1)  # Stagger posts
            
    except Exception as e:
        logging.error(f"Error: {e}")
        time.sleep(5)

{'id': 'fd00a637-bb2e-491d-aeaf-d9f311cb0fc8', 'user_id': 'herberto43', 'text': 'Just received my order and the quality is outstanding! Fast shipping too 📦✨', 'timestamp': 1757929489.5020554, 'domain': 'retail', 'user_profile': {'age_group': '55+', 'follower_count': 43217, 'verified': False, 'location': 'Huelva', 'language': 'ja'}, 'true_sentiment': {'polarity': 'positive', 'confidence': 0.8209313485196799, 'emotions': {'joy': 0, 'anger': 7, 'fear': 10, 'sadness': 7, 'surprise': 0, 'disgust': 0}, 'primary_emotion': 'neutral', 'is_sarcastic': False, 'aspects': {'product': 0.7676493524605925, 'shipping': 0.24624408897142686}}, 'engagement': {'likes': 975, 'shares': 87, 'comments': 49, 'reach': 74615}, 'features': {'text_length': 75, 'has_emojis': False, 'has_hashtags': False, 'has_mentions': False, 'has_urls': False, 'sentiment_words_count': {'positive': 0, 'negative': 0}}}
{'id': '5c395ed9-3c36-4e15-a577-0751c0d22752', 'user_id': 'victor61', 'text': 'Hidden fees everywhere! This bank is

ERROR:root:Error: 'mixed'


{'id': '4315487a-bdc5-406c-90ec-3d8dc519131e', 'user_id': 'jacquesleclercq', 'text': 'Sizing is completely wrong. Waste of money 👎', 'timestamp': 1757929499.510173, 'domain': 'retail', 'user_profile': {'age_group': '46-55', 'follower_count': 1438, 'verified': False, 'location': 'South Nicholas', 'language': 'ja'}, 'true_sentiment': {'polarity': 'negative', 'confidence': 0.6598344654968771, 'emotions': {'joy': 4, 'anger': 0, 'fear': 4, 'sadness': 9, 'surprise': 0, 'disgust': 0}, 'primary_emotion': 'neutral', 'is_sarcastic': False, 'aspects': {'price': 0.8958573311684636}}, 'engagement': {'likes': 352, 'shares': 52, 'comments': 23, 'reach': 3064}, 'features': {'text_length': 44, 'has_emojis': True, 'has_hashtags': False, 'has_mentions': False, 'has_urls': False, 'sentiment_words_count': {'positive': 0, 'negative': 0}}}
{'id': 'f41972eb-b9c2-42b2-8c39-c309633fce8f', 'user_id': 'georges24', 'text': 'This investment app made portfolio management so much easier! Great UX 📈', 'timestamp': 175

ERROR:root:Error: 'mixed'


{'id': 'b6a8a17f-0a51-4ade-8c3e-7d2bcffbb263', 'user_id': 'heinz-dieterheinrich', 'text': 'Product looks nothing like the website photos. Very misleading 😤 #finance #fashion', 'timestamp': 1757929508.5157168, 'domain': 'retail', 'user_profile': {'age_group': '55+', 'follower_count': 8101, 'verified': False, 'location': 'Jenniferchester', 'language': 'de'}, 'true_sentiment': {'polarity': 'negative', 'confidence': 0.9196903464553285, 'emotions': {'joy': 9, 'anger': 0, 'fear': 3, 'sadness': 9, 'surprise': 0, 'disgust': 4}, 'primary_emotion': 'neutral', 'is_sarcastic': False, 'aspects': {}}, 'engagement': {'likes': 443, 'shares': 75, 'comments': 28, 'reach': 23038}, 'features': {'text_length': 82, 'has_emojis': False, 'has_hashtags': True, 'has_mentions': False, 'has_urls': False, 'sentiment_words_count': {'positive': 0, 'negative': 0}}}
{'id': '26fba272-02ed-40fe-a6e6-719b270fee50', 'user_id': 'raymond04', 'text': 'Customer service has no clue about their own products. Frustrated! 🤦', 'ti

...

{'id': '28893fb3-a27e-4c4a-aa80-fbe2b5d0be5e', 'user_id': 'yoichiinoue', 'text': 'Love the design but quality could be better for this price point 🤔', 'timestamp': 1757929769.7919126, 'domain': 'retail', 'user_profile': {'age_group': '26-35', 'follower_count': 26759, 'verified': False, 'location': 'Pontevedra', 'language': 'fr'}, 'true_sentiment': {'polarity': 'mixed', 'confidence': 0.7282298653382613, 'emotions': {'joy': 8, 'anger': 0, 'fear': 0, 'sadness': 10, 'surprise': 9, 'disgust': 0}, 'primary_emotion': 'neutral', 'is_sarcastic': False, 'aspects': {'product': 0.39846784960504467, 'price': 0.14914441501227327}}, 'engagement': {'likes': 389, 'shares': 95, 'comments': 25, 'reach': 52126}, 'features': {'text_length': 66, 'has_emojis': True, 'has_hashtags': False, 'has_mentions': False, 'has_urls': False, 'sentiment_words_count': {'positive': 1, 'negative': 0}}}
{'id': '39ec2295-d49b-4885-9666-6e00d4383539', 'user_id': 'ekondo', 'text': 'Customer service has no clue about their own p

ERROR:root:Error: 'mixed'


{'id': 'ee2d3c38-59db-4dcb-b52d-343c758820c4', 'user_id': 'alphonse27', 'text': 'Product is okay, not great but not terrible either. Average experience 🤷', 'timestamp': 1757929778.7993464, 'domain': 'retail', 'user_profile': {'age_group': '46-55', 'follower_count': 22925, 'verified': False, 'location': '三宅島三宅村', 'language': 'ja'}, 'true_sentiment': {'polarity': 'mixed', 'confidence': 0.9444672247778461, 'emotions': {'joy': 2, 'anger': 1, 'fear': 8, 'sadness': 9, 'surprise': 6, 'disgust': 0}, 'primary_emotion': 'neutral', 'is_sarcastic': False, 'aspects': {}}, 'engagement': {'likes': 262, 'shares': 45, 'comments': 44, 'reach': 29256}, 'features': {'text_length': 72, 'has_emojis': False, 'has_hashtags': False, 'has_mentions': False, 'has_urls': False, 'sentiment_words_count': {'positive': 1, 'negative': 1}}}
{'id': '8b5efb20-28b0-4fdf-b9eb-7aa9cb14bdca', 'user_id': 'josefineritter', 'text': 'Finally found a bank with reasonable fees. Switching was worth it 💰', 'timestamp': 1757929779.800

ERROR:root:Error: 'mixed'


KeyboardInterrupt: 

In [8]:
len(user_posts)

193

In [9]:
user_posts

[{'id': 'fd00a637-bb2e-491d-aeaf-d9f311cb0fc8',
  'user_id': 'herberto43',
  'text': 'Just received my order and the quality is outstanding! Fast shipping too 📦✨',
  'timestamp': 1757929489.5020554,
  'domain': 'retail',
  'user_profile': {'age_group': '55+',
   'follower_count': 43217,
   'verified': False,
   'location': 'Huelva',
   'language': 'ja'},
  'true_sentiment': {'polarity': 'positive',
   'confidence': 0.8209313485196799,
   'emotions': {'joy': 0,
    'anger': 7,
    'fear': 10,
    'sadness': 7,
    'surprise': 0,
    'disgust': 0},
   'primary_emotion': 'neutral',
   'is_sarcastic': False,
   'aspects': {'product': 0.7676493524605925,
    'shipping': 0.24624408897142686}},
  'engagement': {'likes': 975, 'shares': 87, 'comments': 49, 'reach': 74615},
  'features': {'text_length': 75,
   'has_emojis': False,
   'has_hashtags': False,
   'has_mentions': False,
   'has_urls': False,
   'sentiment_words_count': {'positive': 0, 'negative': 0}}},
 {'id': '5c395ed9-3c36-4e15-a57

In [13]:
pip install torch

   ---------------------------------------- 0.0/241.3 MB ? eta -:--:--
    --------------------------------------- 4.2/241.3 MB 25.2 MB/s eta 0:00:10
   - -------------------------------------- 9.2/241.3 MB 23.8 MB/s eta 0:00:10
   -- ------------------------------------- 14.4/241.3 MB 24.5 MB/s eta 0:00:10
   --- ------------------------------------ 18.4/241.3 MB 23.6 MB/s eta 0:00:10
   --- ------------------------------------ 22.8/241.3 MB 22.5 MB/s eta 0:00:10
   ---- ----------------------------------- 26.7/241.3 MB 22.0 MB/s eta 0:00:10
   ---- ----------------------------------- 29.9/241.3 MB 20.8 MB/s eta 0:00:11
   ----- ---------------------------------- 33.0/241.3 MB 20.4 MB/s eta 0:00:11
   ------ --------------------------------- 36.4/241.3 MB 20.0 MB/s eta 0:00:11
   ------ --------------------------------- 39.8/241.3 MB 19.5 MB/s eta 0:00:11
   ------- -------------------------------- 43.3/241.3 MB 19.1 MB/s eta 0:00:11
   ------- -------------------------------- 46.9/24

In [14]:
import json
import numpy as np
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle
import logging
import re
from collections import Counter

In [ ]:
class AdvancedSentimentAnalyzer:
    def __init__(self):
        self.models = self.load_models()
        self.domain_models = self.load_domain_specific_models()
        self.sarcasm_detector = self.load_sarcasm_detector()
        self.emotion_analyzer = self.load_emotion_analyzer()

    def load_models(self):
        models = {}
        try:
            models['bert'] = pipeline(
                "sentiment-analysis",
                model = "cardiffnlp/twitter-roberta-base-sentiment-latest",
                device = 0 if torch.cude.is_available() else -1
            )

            models['multilingual'] = pipeline(
                "sentiment-analysis",
                model = "nlptown/bert-base-multilingual-uncased-sentiment",
                device = 0 if torch.cuda.is_available() else -1
            )

            logging.info("Successfully loaded transformer models")
        except Exception as e:
            logging.error(f"Error loading models: {e}")
            models = self.create_fallback_models()
        return models 
    
    def load_domain_specific_models(self):
        return {
            'retail': {'model_path': '/models/retail_sentiment.pkl', 'accuracy': 0.89},
            'finance': {'model_path': '/models/finance_sentiment.pkl', 'accuracy': 0.85},
            'healthcare': {'model_path': '/models/healthcare_sentiment.pkl', 'accuracy': 0.87}
        }
    
    def load_sarcash_detector(self):
        return pipeline(
            "text-classification",
            model = "helinivan/english-sarcasm-detector",
            device = 0 if torch.cude.is_available() else -1
        )
    
    def load_emotion_analyzer(self):
        return pipeline(
            "text-classification",
            model = "j-hartmann/emotion-english-distilroberta-base",
            device = 0 if torch.cuda.is_available() else -1

        )
    
    def preprocess_text(self, text):
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags = re.MULTILINE)
        text = ' '.join(text.split())
        text = self.handle_negations(text)
        return text
    
    def handle_negations(self, text):
        negation_words = ['not', 'no', 'never', 'nothing', 'nobody', 'nowhere', 'neither', 'nor']
        words = text.split()

        for i, word in enumerate(words):
            if word.lower() in negation_words and i+1 < len(words):
                words[i+1] = f"NOT_{words[i+1]}"
        return ' '.join(words)
    
    def analyze_sentiment_ensemble(self, text, domain = 'general'):
        results = {}
        try:
            if 'bert' in self.models:
                bert_result = self.model['bert'][text][0]
                results['bert'] = {
                    'label':bert_result['label'],
                    'score':bert_result['score']
                }
            if domain in self.domain_models:
                domain_score = self.simulate_domain_prediction(text,domain)
                results['domain'] = domain_score

            if 'multilingual' in self.models:
                multi_result = self.model['multilingual'][text][0]
                results['multilingual'] = {
                    'label': multi_result['label'],
                    'score': multi_result['score']
                }
            final_prediction = self.ensemble_predictions(results)
        except Exception as e:
            logging.error(f"Error in sentiment analysis: {e}")
            final_prediction = {'label': 'NEUTRAL', 'confidence': 0.5}
        return final_prediction
    

    def simulate_domain_prediction(self, text, domain):
        domain_keywords = {
            'retail': ['product', 'quality', 'shipping', 'price', 'customer service'],
            'finance': ['fees', 'rates', 'investment', 'returns', 'advisor'],
            'healthcare': ['doctor', 'treatment', 'insurance', 'appointment', 'medication']
        }

        positive_words = ['good', 'great', 'excellent', 'amazing', 'perfect']
        negative_words = ['bad', 'terrible', 'awful', 'horrible', 'worst']

        text_lower = text.lower()
        pos_score = sum(1 for word in positive_words if word in text_lower)
        neg_score = sum(1 for word in negative_words if word in text_lower)

        if pos_score > neg_score:
            return {'label': 'POSITIVE', 'score': 0.7 * pos_score * 0.1}
        elif neg_score > pos_score:
            return {'label': 'NEGATIVE', 'score': 0.7 * neg_score *0.1}
        else:
            return {'label': 'NEUTRAL', 'score': 0.6}
        
    
    def ensemble_predictions(self, results):
        if not results:
            return {'label': 'NEUTRAL', 'confidence': 0.5}
        
        weights = {
            'bert':0.4,
            'domain':0.3,
            'multilingual':0.3
        }

        label_scores = {'POSITIVE':0,'NEGATIVE':0,'NEUTRAL':0}
        total_weight = 0

        for model_name, result in results.items():
            if model_name in weights:
                weight = weights[model_name]
                label = result['label'].upper()

                if 'POS' in label or label == '5 STARS' or label == 'LABEL_2':
                    label = 'POSITIVE'
                elif 'NEG' in label or label == '1 STAR' or label == 'LABEL_0':
                    label = 'NEGATIVE'
                else:
                    label = 'NEUTRAL'

                score= result.get('score', 0.5)

    

















